# AI Challenge Stage 1 — сегментация области манипуляции

По одному RGB-изображению предсказываем бинарную маску области, которая была изменена
(добавлен или удалён объект, инпейнт, локальная правка).

**Метрика — AIC Score**, гармоническое среднее двух величин:

- `Dice_pos` — средний Dice по изображениям с изменениями;
- `FPR_neg` — доля чистых изображений, на которых площадь предсказанной маски занимает
  не меньше 1% кадра (ложная тревога).

```
AIC = 2 * Dice_pos * (1 - FPR_neg) / (Dice_pos + (1 - FPR_neg))
```

Решение: U-Net (ResNet34) с дополнительным шумовым каналом. Порог бинаризации
подбирается по AIC на валидации. Все данные загружаются заранее и распаковываются
в корень проекта (см. README.md).

## 1. Требования

- Python 3.11 (у torch нет колёс для 3.14);
- GPU с CUDA (например, RTX 3050 и выше);
- распакованные данные конкурса в корне проекта.

## 2. Установка зависимостей

Все дополнительные библиотеки устанавливаются прямо из ноутбука. Запустите эту
ячейку один раз; повторный запуск безопасен (pip пропустит уже установленное).

In [ ]:
%pip install -q "torch==2.5.1+cu121" "torchvision==0.20.1+cu121" --index-url https://download.pytorch.org/whl/cu121
%pip install -q -r requirements.txt

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))

## 3. Пути и параметры запуска

Корень проекта ищется автоматически по маркеру `src/config.py`, поэтому ноутбук
можно запускать из любого каталога.

In [ ]:
import os
import sys
from pathlib import Path

def find_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "config.py").exists():
            return candidate
    raise RuntimeError("Не найден корень проекта (нет src/config.py)")

ROOT = find_root()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("корень проекта:", ROOT)

# Параметры, воспроизводящие отправленное решение.
params = dict(
    encoder="resnet34",
    img_size=256,
    batch_size=8,
    epochs=10,
    name="manip_unet_resnet34",
)

## 4. Подготовка разметки

Скрипт `src/prepare.py` читает все маски и кэширует долю площади правки. Это нужно,
чтобы отделить настоящие позитивы от "чистых" строк с пустой маской — последние
используются как негативы для составляющей FPR метрики.

In [ ]:
import subprocess

result = subprocess.run([sys.executable, "-m", "src.prepare"], cwd=ROOT)
print("exit code:", result.returncode)

## 5. Обучение

Скрипт разбивает данные по группам масок (без утечки), добавляет негативы, проверяет
лимит 100 GFLOPs и в конце подбирает порог бинаризации по AIC. Чекпоинты сохраняются
в `outputs/checkpoints/`.

In [ ]:
cmd = [
    sys.executable, "-m", "src.train",
    "--encoder", params["encoder"],
    "--img-size", str(params["img_size"]),
    "--batch-size", str(params["batch_size"]),
    "--epochs", str(params["epochs"]),
    "--name", params["name"],
]
print("команда:", " ".join(cmd))
result = subprocess.run(cmd, cwd=ROOT)
print("exit code:", result.returncode)

## 6. Инференс и сборка посылки

Модель предсказывает маски для тестовых изображений, возвращает их к исходному
разрешению, бинаризует подобранным порогом и упаковывает в `submission.zip`.

In [ ]:
checkpoint = ROOT / "outputs" / "checkpoints" / f"{params['name']}_best.pth"
threshold_json = ROOT / "outputs" / "checkpoints" / f"{params['name']}_threshold.json"

cmd = [
    sys.executable, "-m", "src.predict",
    "--encoder", params["encoder"],
    "--img-size", str(params["img_size"]),
    "--checkpoint", str(checkpoint),
    "--threshold-json", str(threshold_json),
]
print("команда:", " ".join(cmd))
result = subprocess.run(cmd, cwd=ROOT)
print("exit code:", result.returncode)

## 7. Проверка посылки

Убеждаемся, что архив собран корректно: 2160 масок, все PNG бинарные (0/255),
каждая маска в разрешении своего изображения.

In [ ]:
import csv
import zipfile
import numpy as np
from PIL import Image

zip_path = ROOT / "outputs" / "submission.zip"
with zipfile.ZipFile(zip_path) as zf:
    names = zf.namelist()
    csv_names = [n for n in names if n.endswith(".csv")]
    png_names = [n for n in names if n.endswith(".png")]
    print("файлов в архиве:", len(names), "| масок:", len(png_names))

    with zf.open(csv_names[0]) as f:
        rows = list(csv.DictReader(f))
    print("строк в submission.csv:", len(rows))
    print("колонки:", list(rows[0].keys()))

    # Проверяем бинарность первой маски.
    with zf.open(png_names[0]) as f:
        mask = np.asarray(Image.open(f))
    print("форма маски:", mask.shape, "| значения:", sorted(set(mask.flatten().tolist()))[:5])

Готово. Итоговая посылка — `outputs/submission.zip`, её содержимое соответствует
формату конкурса: `submission.csv` плюс папка `predictions/` с бинарными PNG-масками.